In [82]:
import polars as pl
import numpy as np
import re
from datetime import date

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(100)

device = torch.device("mps") if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

Device: mps


In [83]:
tech_path = "../data/model_staging/tech_modeling_table.parquet"
df_tech = pl.read_parquet(tech_path)
print(f"Tech table shape: {df_tech.shape}")

Tech table shape: (24048, 239)


In [84]:
# Target: car_t2_t10 = cumulative abnormal return over t+2 to t+10 (post-earnings drift)
# Features: full window t=-10 to t=+1 (no leakage since target starts at t+2)
df_tech['car_t2_t10'].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.7, 0.75, 0.9, 0.95, 0.99])

statistic,value
str,f64
"""count""",24048.0
"""null_count""",0.0
"""mean""",0.001891
"""std""",0.049086
"""min""",-0.747742
"""1%""",-0.122485
"""5%""",-0.070244
"""10%""",-0.050763
"""25%""",-0.024126


In [91]:
df_tech

symbol,earnings_date,entry_price,target_return,target_class,car_3day,car_t2_t10,max_high,min_high,max_day,min_day,open_pct_t-10,high_pct_t-10,low_pct_t-10,volume_rel_t-10,rsi_t-10,macd_t-10,macd_hist_t-10,roc_t-10,ema50_pct_t-10,ema200_pct_t-10,ema50_200_pct_t-10,adx_t-10,atr_t-10,bb_width_t-10,bb_pct_b_t-10,sigma_t-10,obv_zscore_t-10,vwap_pct_t-10,vix_close_t-10,open_pct_t-9,high_pct_t-9,low_pct_t-9,volume_rel_t-9,rsi_t-9,macd_t-9,macd_hist_t-9,roc_t-9,ema50_pct_t-9,ema200_pct_t-9,ema50_200_pct_t-9,adx_t-9,atr_t-9,bb_width_t-9,bb_pct_b_t-9,sigma_t-9,obv_zscore_t-9,vwap_pct_t-9,vix_close_t-9,open_pct_t-8,…,roc_t-1,ema50_pct_t-1,ema200_pct_t-1,ema50_200_pct_t-1,adx_t-1,atr_t-1,bb_width_t-1,bb_pct_b_t-1,sigma_t-1,obv_zscore_t-1,vwap_pct_t-1,vix_close_t-1,open_pct_t0,high_pct_t0,low_pct_t0,volume_rel_t0,rsi_t0,macd_t0,macd_hist_t0,roc_t0,ema50_pct_t0,ema200_pct_t0,ema50_200_pct_t0,adx_t0,atr_t0,bb_width_t0,bb_pct_b_t0,sigma_t0,obv_zscore_t0,vwap_pct_t0,vix_close_t0,open_pct_t+1,high_pct_t+1,low_pct_t+1,volume_rel_t+1,rsi_t+1,macd_t+1,macd_hist_t+1,roc_t+1,ema50_pct_t+1,ema200_pct_t+1,ema50_200_pct_t+1,adx_t+1,atr_t+1,bb_width_t+1,bb_pct_b_t+1,sigma_t+1,obv_zscore_t+1,vwap_pct_t+1,vix_close_t+1
str,date,f64,f64,i64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""LYB""",2020-07-31,42.475105,0.183004,2,-0.099336,0.084218,50.248203,44.539732,7,2,0.008175,0.017518,-0.006131,-0.466738,56.244829,0.011621,0.002661,2.299903,0.060346,-0.007481,-0.063967,10.51737,0.040626,0.100177,0.794481,0.020628,-0.306764,0.047946,25.68,0.019012,0.032036,-0.002545,-0.432572,51.672713,0.010835,0.001317,-2.12451,0.032652,-0.031804,-0.062418,10.024723,0.041154,0.096452,0.520329,0.021298,-0.411784,0.022934,24.459999,-0.008227,…,-5.184418,-0.00462,-0.052683,-0.048286,8.06714,0.037747,0.075325,0.000582,0.019819,-0.121901,-0.027162,24.76,0.026392,0.034389,-0.015995,0.668771,39.213279,0.001579,-0.008466,-8.72993,-0.045096,-0.092035,-0.049156,9.797605,0.040911,0.105249,-0.219292,0.022621,-0.392894,-0.063225,24.459999,0.014808,0.02704,-0.000322,0.344674,38.427014,-0.004488,-0.011676,-6.991045,-0.049149,-0.096821,-0.050136,11.404466,0.040181,0.135342,-0.067169,0.021281,-0.61579,-0.063568,24.280001
"""LYB""",2023-10-27,76.03109,0.036976,1,0.019794,-0.022796,78.842392,76.131806,10,3,0.003776,0.014674,-0.0041,-0.323695,42.163044,-0.013228,-0.001008,-2.133043,-0.023657,0.015104,0.039701,30.334723,0.022799,0.054506,0.389251,0.013491,0.762318,-0.018973,19.32,0.001927,0.007173,-0.006745,-0.421522,45.343562,-0.012117,0.000006,-0.042806,-0.01535,0.022865,0.038811,30.418959,0.022076,0.054363,0.536082,0.012991,0.814194,-0.007018,17.209999,-0.022797,…,-2.75942,-0.043783,-0.01634,0.028699,32.182988,0.021793,0.067468,0.098913,0.014806,0.667164,-0.027077,20.68,0.001769,0.009951,-0.021893,0.495901,41.020603,-0.015152,-0.002034,-2.416901,-0.036189,-0.009893,0.027284,33.333808,0.022381,0.07041,0.231914,0.013056,0.800155,-0.017975,21.27,0.008057,0.010265,-0.02064,0.147614,41.6783,-0.014836,-0.001394,-3.008253,-0.033177,-0.008061,0.025978,34.210997,0.022954,0.069164,0.289271,0.012625,0.899396,-0.01437,19.75
"""LYB""",2021-04-30,76.533699,0.082507,2,0.013232,0.061157,82.848248,77.569582,6,2,0.00623,0.014877,-0.006137,-0.281581,58.323214,0.007363,0.001179,1.894857,0.051681,0.227514,0.167193,10.782928,0.027548,0.050157,0.983224,0.011011,1.457469,0.027006,16.25,0.005037,0.007929,-0.0125,-0.414743,57.146784,0.008037,0.001466,2.86919,0.046278,0.220804,0.166806,10.50057,0.027123,0.053627,0.851412,0.0104,1.387482,0.026602,17.290001,0.019973,…,0.364173,0.040889,0.205571,0.158213,8.608718,0.027361,0.06186,0.753792,0.021437,1.437983,0.022441,17.610001,0.017158,0.03

In [85]:
df_tech['car_3day'].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.7, 0.75, 0.9, 0.95, 0.99])

statistic,value
str,f64
"""count""",24048.0
"""null_count""",0.0
"""mean""",0.003473
"""std""",0.061556
"""min""",-0.538946
"""1%""",-0.160619
"""5%""",-0.09072
"""10%""",-0.064799
"""25%""",-0.029435


In [86]:
fund_path = "../data/model_staging/fundamentalIndicators/modeling_fundamentals.parquet"
df_fund = pl.read_parquet(fund_path)
df_fund = df_fund.select([
    "symbol", pl.col("reportedDate").alias("earnings_date"),
    "eps_growth_qoq", "revenue_growth_qoq",
    "gross_margin", "gross_margin_qoq",
    "debt_to_equity", "debt_to_equity_qoq",
    "fcf_margin", "fcf_margin_qoq",
    "roe", "roe_qoq"
])

df_finbert = pl.read_parquet("../data/model_staging/finbert_tx_agg_weighted.parquet")
df_finbert = df_finbert.select([
    pl.col("symbol"), pl.col("reportedDate").alias("earnings_date"),
    "pos_prob", "neg_prob"
])

df_nz = pl.read_parquet("../data/model_staging/nz_sentiment.parquet")
df_nz = df_nz.select([
    pl.col("symbol"), pl.col("reportedDate").alias("earnings_date"),
    "overall_sentiment_score_pre", "ticker_sentiment_score_pre",
    "overall_sentiment_score_post", "ticker_sentiment_score_post",
])

df_model = df_tech.join(df_fund, on=["symbol", "earnings_date"], how="left")
df_model = df_model.join(df_finbert, on=["symbol", "earnings_date"], how="left", suffix="_fb")
df_model = df_model.join(df_nz, on=["symbol", "earnings_date"], how="left", suffix="_nz")

df_sector_raw = pl.read_parquet("../data/model_staging/finbert_tx_agg_weighted.parquet")
df_sector = df_sector_raw.select(["symbol", "sector"]).unique()
df_sector = df_sector.with_columns(pl.col("sector").fill_null("Unknown"))
sectors = sorted(df_sector["sector"].unique().to_list())
df_sector = df_sector.with_columns([
    (pl.col("sector") == s).cast(pl.Int8).alias(f"sector_{s.replace(' ', '_')}")
    for s in sectors
]).drop("sector")
df_model = df_model.join(df_sector, on="symbol", how="left")

print(f"Final shape: {df_model.shape}")

Final shape: (24048, 267)


In [87]:
exclude_cols = ["symbol", "earnings_date", "target_return", "target_class",
                "entry_price", "max_high", "min_high", "max_day", "min_day",
                "car_3day", "car_t2_t10"]
feature_cols = [c for c in df_model.columns if c not in exclude_cols]

time_cols = [c for c in feature_cols if re.match(r".+_t[+-]?\d+$", c)]
static_cols = [c for c in feature_cols if c not in time_cols]

bases = sorted(set(re.sub(r"_t[+-]?\d+$", "", c) for c in time_cols))
steps = sorted({int(re.search(r"_t([+-]?\d+)$", c).group(1)) for c in time_cols})
n_steps, n_bases, n_static = len(steps), len(bases), len(static_cols)
print(f"Time steps: {steps}  ({n_steps} steps x {n_bases} bases = {n_steps * n_bases})")
print(f"Static features: {n_static}")

Time steps: [-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1]  (12 steps x 19 bases = 228)
Static features: 28


In [88]:
class LSTMRegressor(nn.Module):
    def __init__(self, n_bases, n_static, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_bases, hidden, batch_first=True)
        self.fc = nn.Linear(hidden + n_static, 1)

    def forward(self, x_seq, x_stat):
        _, (hn, _) = self.lstm(x_seq)
        h = hn[-1]
        if x_stat is not None:
            h = torch.cat([h, x_stat], dim=1)
        return self.fc(h).squeeze(-1)


class BiLSTMRegressor(nn.Module):
    def __init__(self, n_bases, n_static, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_bases, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden * 2 + n_static, 1)

    def forward(self, x_seq, x_stat):
        _, (hn, _) = self.lstm(x_seq)
        h = torch.cat((hn[-2], hn[-1]), dim=1)
        if x_stat is not None:
            h = torch.cat([h, x_stat], dim=1)
        return self.fc(h).squeeze(-1)


class CNN_LSTMRegressor(nn.Module):
    def __init__(self, n_bases, n_static, hidden=64, kernel=3):
        super().__init__()
        self.conv = nn.Conv1d(n_bases, n_bases, kernel, padding=kernel//2)
        self.lstm = nn.LSTM(n_bases, hidden, batch_first=True)
        self.fc = nn.Linear(hidden + n_static, 1)

    def forward(self, x_seq, x_stat):
        x_seq = self.conv(x_seq.transpose(1, 2)).transpose(1, 2)
        _, (hn, _) = self.lstm(x_seq)
        h = hn[-1]
        if x_stat is not None:
            h = torch.cat([h, x_stat], dim=1)
        return self.fc(h).squeeze(-1)

In [89]:
Train_Window = 7
Folds = [
    (date(y - Train_Window, 1, 1), date(y, 1, 1), date(y + 1, 1, 1))
    for y in range(2021, 2026)
]

# Full feature window (t=-10 to t=+1) -- no leakage since target starts at t+2
cols_for_grid = time_cols + static_cols
col_to_idx = {c: i for i, c in enumerate(cols_for_grid)}
base_to_idx = {b: i for i, b in enumerate(bases)}
step_to_idx = {s: i for i, s in enumerate(steps)}
col_grid = np.full((n_steps, n_bases), -1, dtype=int)
for c in time_cols:
    s = int(re.search(r"_t([+-]?\d+)$", c).group(1))
    b = re.sub(r"_t[+-]?\d+$", "", c)
    col_grid[step_to_idx[s], base_to_idx[b]] = col_to_idx[c]


def reshape_features(df):
    X = df.select(cols_for_grid).to_numpy().astype(np.float32)
    X = np.where(np.isinf(X), np.nan, X)
    imputer = SimpleImputer(strategy="median")
    X = imputer.fit_transform(X)
    X_time = np.zeros((X.shape[0], n_steps, n_bases), dtype=np.float32)
    for s in range(n_steps):
        for b in range(n_bases):
            idx = col_grid[s, b]
            if idx >= 0:
                X_time[:, s, b] = X[:, idx]
    X_static = X[:, [col_to_idx[c] for c in static_cols]] if n_static > 0 else None
    scaler = StandardScaler()
    X_time = scaler.fit_transform(X_time.reshape(-1, n_bases)).reshape(X_time.shape)
    if X_static is not None:
        s_scaler = StandardScaler()
        X_static = s_scaler.fit_transform(X_static)
    return X_time, X_static, imputer, scaler, (s_scaler if n_static > 0 else None)


def transform_test(df, imputer, scaler, s_scaler):
    X = df.select(cols_for_grid).to_numpy().astype(np.float32)
    X = np.where(np.isinf(X), np.nan, X)
    X = imputer.transform(X)
    X_time = np.zeros((X.shape[0], n_steps, n_bases), dtype=np.float32)
    for s in range(n_steps):
        for b in range(n_bases):
            idx = col_grid[s, b]
            if idx >= 0:
                X_time[:, s, b] = X[:, idx]
    X_time = scaler.transform(X_time.reshape(-1, n_bases)).reshape(X_time.shape)
    X_static = X[:, [col_to_idx[c] for c in static_cols]] if n_static > 0 else None
    if X_static is not None and s_scaler is not None:
        X_static = s_scaler.transform(X_static)
    return X_time, X_static


BATCH_SIZE = 128
EPOCHS = 30
LR = 1e-3

results = {}

for ModelClass, name in [(LSTMRegressor, "LSTM"), (BiLSTMRegressor, "BiLSTM"),
                          (CNN_LSTMRegressor, "CNN-LSTM")]:
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    clf = {"fold_rmse": [], "fold_mae": [], "fold_da": [], "preds": [], "true": []}

    for fold_num, (train_start, test_start, test_end) in enumerate(Folds, 1):
        train = df_model.filter(
            (pl.col("earnings_date") >= train_start) & (pl.col("earnings_date") < test_start)
        )
        test = df_model.filter(
            (pl.col("earnings_date") >= test_start) & (pl.col("earnings_date") < test_end)
        )

        X_train_t, X_train_s, imputer, scaler, s_scaler = reshape_features(train)
        X_test_t, X_test_s = transform_test(test, imputer, scaler, s_scaler)

        # Target: car_t2_t10 in percentage scale (match paper's convention)
        y_train = train["car_t2_t10"].to_numpy().astype(np.float32) * 100
        y_test = test["car_t2_t10"].to_numpy().astype(np.float32) * 100

        train_ds = TensorDataset(
            torch.tensor(X_train_t),
            torch.tensor(X_train_s) if X_train_s is not None else torch.zeros(len(X_train_t), 1),
            torch.tensor(y_train),
        )
        test_ds = TensorDataset(
            torch.tensor(X_test_t),
            torch.tensor(X_test_s) if X_test_s is not None else torch.zeros(len(X_test_t), 1),
            torch.tensor(y_test),
        )

        train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False)

        model = ModelClass(n_bases, n_static).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.MSELoss()

        for epoch in range(EPOCHS):
            model.train()
            for x_seq, x_stat, y in train_loader:
                x_seq, x_stat, y = x_seq.to(device), x_stat.to(device), y.to(device)
                opt.zero_grad()
                loss = loss_fn(model(x_seq, x_stat if n_static > 0 else None), y)
                loss.backward()
                opt.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for x_seq, x_stat, y in test_loader:
                x_seq, x_stat = x_seq.to(device), x_stat.to(device)
                pred = model(x_seq, x_stat if n_static > 0 else None)
                preds.append(pred.cpu().numpy())
                trues.append(y.numpy())
        preds = np.concatenate(preds)
        trues = np.concatenate(trues)

        rmse = float(np.sqrt(mean_squared_error(trues, preds)))
        mae = float(mean_absolute_error(trues, preds))
        da = float(np.mean((np.sign(preds) == np.sign(trues))
                          | ((preds == 0) & (trues == 0))))

        clf["fold_rmse"].append(rmse)
        clf["fold_mae"].append(mae)
        clf["fold_da"].append(da)
        clf["preds"].extend(preds)
        clf["true"].extend(trues)
        print(f"  Fold {fold_num} [{test_start.year}]: RMSE={rmse:.3f}, MAE={mae:.3f}, DA={da:.3f}")

    results[name] = clf


LSTM
  Fold 1 [2021]: RMSE=5.413, MAE=3.904, DA=0.517
  Fold 2 [2022]: RMSE=7.282, MAE=5.167, DA=0.510
  Fold 3 [2023]: RMSE=4.858, MAE=3.505, DA=0.490
  Fold 4 [2024]: RMSE=5.337, MAE=3.839, DA=0.512
  Fold 5 [2025]: RMSE=6.043, MAE=4.463, DA=0.514

BiLSTM
  Fold 1 [2021]: RMSE=5.457, MAE=4.029, DA=0.490
  Fold 2 [2022]: RMSE=7.015, MAE=4.979, DA=0.520
  Fold 3 [2023]: RMSE=5.170, MAE=3.707, DA=0.483
  Fold 4 [2024]: RMSE=5.454, MAE=3.934, DA=0.503
  Fold 5 [2025]: RMSE=6.353, MAE=4.647, DA=0.496

CNN-LSTM
  Fold 1 [2021]: RMSE=5.426, MAE=3.970, DA=0.501
  Fold 2 [2022]: RMSE=6.693, MAE=4.749, DA=0.489
  Fold 3 [2023]: RMSE=4.715, MAE=3.419, DA=0.473
  Fold 4 [2024]: RMSE=5.207, MAE=3.788, DA=0.519
  Fold 5 [2025]: RMSE=6.368, MAE=4.636, DA=0.490


In [90]:
print("Table 4. Comparison with existing multimodal deep learning models.")
print(f"{'Model':<15} {'RMSE':>8} {'MAE':>8} {'DA (%)':>8}")
print("-" * 42)
for name in ["LSTM", "BiLSTM", "CNN-LSTM"]:
    r = results[name]
    true = np.array(r["true"])
    preds = np.array(r["preds"])
    avg_rmse = np.mean(r["fold_rmse"])
    avg_mae = np.mean(r["fold_mae"])
    da_pooled = float(np.mean((np.sign(preds) == np.sign(true))
                             | ((preds == 0) & (true == 0))))
    print(f"{name:<15} {avg_rmse:>8.3f} {avg_mae:>8.3f} {da_pooled * 100:>8.2f}")

Table 4. Comparison with existing multimodal deep learning models.
Model               RMSE      MAE   DA (%)
------------------------------------------
LSTM               5.787    4.176    50.87
BiLSTM             5.890    4.259    49.85
CNN-LSTM           5.682    4.113    49.44
